In [0]:
 %sql
-- """
-- Year-wise revenue breakdown (2016-2022 showing peak of ₹164K in 2020)
-- """
create or replace view invt.vw_yearly_revenue_breakdown as
SELECT 
  dt.Year,
  SUM(ft.Price * ft.ProductQuantity) AS Revenue
FROM invt.factinventoryproduct AS ft
INNER JOIN invt.dim_date AS dt ON (ft.orderDateID) = (dt.DateID)
WHERE dt.Year BETWEEN 2016 AND 2022
GROUP BY dt.Year
ORDER BY dt.Year;

select * from invt.vw_yearly_revenue_breakdown


In [0]:
%sql
	-- """
    -- Top revenue-generating products (Shoes: ₹212K, Books: ₹73.9K, T-shirts: ₹72.7K)
    -- """
  create or replace view invt.vw_top_revenue_generating_products as 
SELECT 
dp.ProductName,
sum(price * ProductQuantity) as product_revenue
FROM invt.factinventoryproduct as ft
INNER JOIN invt.dim_product as dp on ft.ProductID = dp.ProductID
GROUP BY dp.ProductName
ORDER BY product_revenue DESC LIMIT 5;

select * from invt.vw_top_revenue_generating_products

In [0]:
%sql
	-- """
  -- Product category performance ranking across 12 categories
  -- """
  create or replace view invt.vw_product_category_performance as
SELECT 
  dp.ProductCategory,
  SUM(ft.Price * ft.ProductQuantity) AS product_revenue,
  RANK() OVER (ORDER BY SUM(ft.Price * ft.ProductQuantity) DESC) AS revenue_rank
FROM invt.factinventoryproduct AS ft
INNER JOIN invt.dim_product AS dp 
  ON ft.ProductID = dp.ProductID
GROUP BY dp.ProductCategory
ORDER BY product_revenue DESC LIMIT 12;

select * from invt.vw_product_category_performance

In [0]:
%sql
-- """
-- Customer segmentation by total purchase value
-- """
SELECT 
ds.CustomerName,
Sum(ft.Price * ft.ProductQuantity) as customer_revenue
FROM invt.factinventoryproduct as ft
INNER JOIN invt.dim_customer as ds 
ON ft.CustomerID = ds.CustomerID
GROUP BY ds.CustomerName
ORDER BY customer_revenue DESC 


In [0]:
%sql
-- """
-- Customer segmentation by total purchase value
-- """
create or replace view invt.vw_customer_segmentation as
SELECT 
ds.CustomerName,
Sum(ft.Price * ft.ProductQuantity) as total_purchase_value,
COUNT(ft.ProductQuantity) as total_product_count 
FROM invt.factinventoryproduct as ft
INNER JOIN invt.dim_customer as ds 
ON ft.CustomerID = ds.CustomerID
INNER JOIN invt.dim_product as dpd 
on ft.ProductID = dpd.ProductID
GROUP BY ds.CustomerName
ORDER BY total_purchase_value DESC ;

select * from invt.vw_customer_segmentation


In [0]:
%sql
-- """
-- Geographic distribution analysis by city (enable map visualizations)
-- """
create or replace view invt.vw_geographic_distribution as
SELECT 
ds.CustomerCity,
Sum(ft.Price * ft.ProductQuantity) as city_revenue
FROM invt.factinventoryproduct as ft 
INNER JOIN invt.dim_customer as ds 
ON ft.CustomerID = ds.CustomerID 
GROUP BY ds.CustomerCity
ORDER BY city_revenue DESC;

select * from invt.vw_geographic_distribution


In [0]:
%sql
-- """
-- Gender-based purchasing patterns
-- """
create or replace view invt.vw_gender_based_purchasing_patterns as
SELECT 
dc.CustomerGender,
sum(ft.Price * ft.ProductQuantity) as revenue,
sum(ft.ProductQuantity) as totalQuantity
FROM invt.factinventoryproduct as ft
INNER JOIN invt.dim_customer as dc 
ON ft.CustomerID = dc.CustomerID 
GROUP BY dc.CustomerGender;

select * from invt.vw_gender_based_purchasing_patterns


In [0]:
%sql
-- """
-- Average order value calculations per product
-- """
create or replace view invt.vw_avg_order as
SELECT 
dp.ProductName,
ROUND (avg(ft.orderDateID), 2 )as avg_order
FROM invt.factinventoryproduct as ft
INNER JOIN invt.dim_product as dp 
ON ft.ProductID = dp.ProductID
GROUP BY dp.ProductName
ORDER BY avg_order DESC;

select * from invt.vw_avg_order


In [0]:
%sql

-- """Current inventory value: ₹765,600 across all locations"""
create or replace view invt.vw_currecnt_inventory_value as
SELECT 
  dc.CustomerCity as location,
  concat('₹',
   format_number( SUM(fos.ProductQuantity * fos.Price) /1000 , 2), 'k')
   as inventory_value
FROM  invt.factinventoryproduct as fos 
JOIN invt.dim_customer as dc ON fos.CustomerID = dc.CustomerID
GROUP BY location
ORDER BY inventory_value DESC;

select * from invt.vw_currecnt_inventory_value


In [0]:
%sql
-- Total products in stock: 9.9K units 
create or replace view invt.vw_total_products_in_stock as
SELECT 
concat(
format_number(
    (SUM(ProductQuantity)) / 1000 , 2) 
    ,'K'  
    )
    AS total_products_in_stock
FROM invt.factinventoryproduct;

select * from invt.vw_total_products_in_stock

In [0]:
%sql
-- Warehouse-level inventory distribution (10 locations)
-- Inventory Value	SUM(stock_quantity * product_price)
create or replace view invt.vw_inventory_value as
SELECT 
  dw.wareHouseCity as location,
  CONCAT('₹',
   format_number(
    SUM(fps.ProductQuantity * fps.Price) 
    / 1000, 2)
    ,'k') as inventory_value
FROM invt.factproductstatus as fps
INNER JOIN invt.dim_product as dp 
on fps.ProductID = dp.ProductID
INNER JOIN invt.dim_warehouse as dw 
on fps.WareHouseID = dw.WareHouseID
GROUP BY dw.wareHouseCity
ORDER BY inventory_value DESC
LIMIT 10;

select * from invt.vw_inventory_value

In [0]:
%sql
-- •	Year-over-year revenue comparison (2016-2022 trend analysis)
CREATE OR REPLACE VIEW invt.vw_yoy_revenue as
SELECT 
  dd.Year,
  SUM(ft.Price * ft.ProductQuantity) as revenue,
  concat('₹',
    format_number(SUM(ft.Price * ft.ProductQuantity) / 10000, 2) ,'K')as revenue_IN
FROM invt.factinventoryproduct as ft
JOIN invt.dim_date as dd ON 
  ft.orderDateID = dd.DateID
WHERE dd.Year BETWEEN 2016 AND 2022
GROUP BY dd.Year;

SELECT * FROM invt.vw_yoy_revenue


In [0]:
%sql
-- •	Month-over-month growth calculations with LAG/LEAD functions
CREATE OR REPLACE VIEW invt.vw_monthly_revenue AS
with revenue as (
SELECT 
  dd.Year,
  dd.Month,
  SUM(ft.ProductQuantity * ft.Price) AS revenue
FROM invt.factinventoryproduct AS ft
JOIN invt.dim_date AS dd
  ON ft.orderDateID = dd.DateID
GROUP BY dd.Year, dd.Month
ORDER BY dd.Year, dd.Month
)
SELECT 
  Year,
  Month,
  revenue,
  LAG(revenue) OVER (ORDER BY Year, Month) AS prev_month_revenue,
  ROUND(
    (revenue - LAG(revenue) OVER (ORDER BY Year, Month)) 
    / NULLIF(LAG(revenue) OVER (ORDER BY Year, Month), 0) * 100, 2
  ) AS mom_growth_percent
FROM revenue
ORDER BY Year, Month;

select * from invt.vw_monthly_revenue

In [0]:
%sql
-- • Quarter-over-quarter performance metrics from invt
CREATE OR REPLACE VIEW invt.vw_quarterly_revenue AS
with quater_revenu as(
SELECT 
  dd.Year,
  quarter(dd.date )as Quarter,
  SUM(ft.Price * ft.ProductQuantity) AS revenue
FROM invt.factinventoryproduct AS ft
JOIN invt.dim_date AS dd ON ft.orderDateID = dd.DateID
GROUP BY dd.Year, quarter --(dd.Month)
ORDER BY dd.Year, quarter --(dd.Month)
)

SELECT 
  Year,
  Quarter,
  revenue,
  LAG(revenue) OVER (ORDER BY Year, Quarter) AS prev_quarter_revenue,
  ROUND(
    (revenue - LAG(revenue) OVER (ORDER BY Year, Quarter)) 
    / NULLIF(LAG(revenue) OVER (ORDER BY Year, Quarter), 0) * 100, 2
  ) AS qoq_growth_percentage
FROM quater_revenu
ORDER BY Year, Quarter;

select * from invt.vw_quarterly_revenue

In [0]:
%sql
-- • Moving averages (3-month, 6-month) for smoothing volatile metrics from invt
create or replace view invt.vw_moving_averages as
SELECT
  Year,
  Month,
  revenue,
  ROUND(
    AVG(revenue) OVER (
      ORDER BY Year, Month
      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2
  ) AS ma_3_month,
  ROUND(
    AVG(revenue) OVER (
      ORDER BY Year, Month
      ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
    ), 2
  ) AS ma_6_month
FROM invt.vw_monthly_revenue
ORDER BY Year, Month;

select * from invt.vw_moving_averages

In [0]:
%sql
-- • Cumulative revenue calculations (running totals) from invt
create or replace view  invt.vw_cumulative_revenue as
SELECT
  dd.Year,
  dd.Month,
  SUM(ft.ProductQuantity * ft.Price) AS revenue,
  SUM(SUM(ft.ProductQuantity * ft.Price)) OVER (
    ORDER BY dd.Year, dd.Month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_revenue
FROM invt.factinventoryproduct AS ft
JOIN invt.dim_date AS dd
  ON ft.orderDateID = dd.DateID
GROUP BY dd.Year, dd.Month
ORDER BY dd.Year, dd.Month;

select * from invt.vw_cumulative_revenue

In [0]:
%sql
-- •	Low-stock alert identification (threshold-based detection)
CREATE OR REPLACE VIEW invt.vw_low_stock AS
SELECT 
fps.productQuantity,
dp.productName,
dps.productstatus
FROM invt.factproductstatus as fps
INNER JOIN invt.dim_productstatus as dps on fps.productstatusID = dps.productstatusID
INNER join invt.dim_product as dp on fps.ProductID = dp.ProductID
WHERE dps.productstatus = 'LowStock';

select * from invt.vw_low_stock;

In [0]:
%sql
-- •	Product status tracking: InStock vs LowStock classification
CREATE OR REPLACE VIEW invt.vw_stock_status AS
SELECT 
SUM(CASE WHEN dps.productstatus = 'LowStock' THEN fps.productQuantity ELSE 0 END) 
as lowstock_productquantity,
sum(CASE WHEN dps.productstatus = 'InStock' THEN fps.productQuantity ELSE 0 END) 
as instock_productquantity
FROM invt.factproductstatus as fps
INNER join invt.dim_productstatus as dps on fps.productstatusID = dps.productstatusID
Inner join invt.dim_product as dp on fps.ProductID = dp.ProductID;

select * from invt.vw_stock_status

In [0]:
%sql
CREATE OR REPLACE VIEW invt.vw_InStock AS
SELECT 
  dp.ProductName,
  fps.productQuantity,
  dps.productstatus
FROM invt.factproductstatus AS fps
INNER JOIN invt.dim_productstatus AS dps ON fps.productstatusID = dps.productstatusID
INNER JOIN invt.dim_product AS dp ON fps.ProductID = dp.ProductID
WHERE dps.productstatus = 'InStock';

SELECT * FROM invt.vw_InStock

In [0]:
%sql
-- • Stock-out prediction using historical consumption trends
-- Predict next month's stock-out risk based on average monthly sales and current stock

CREATE OR REPLACE VIEW invt.vw_stock_out_prediction AS
WITH avg_monthly_sales AS (
  SELECT
    dp.ProductID,
    dp.ProductName,
    AVG(ft.ProductQuantity) AS avg_monthly_sales_quantity
  FROM invt.factinventoryproduct AS ft 
  JOIN invt.dim_product AS dp ON ft.ProductID = dp.ProductID
  JOIN invt.dim_date AS dd ON ft.orderDateID = dd.DateID
  GROUP BY dp.ProductID, dp.ProductName
),
current_stock AS (
  SELECT
    fps.ProductID,
    SUM(fps.productQuantity) AS current_quantity
  FROM invt.factproductstatus AS fps
  GROUP BY fps.ProductID
)
SELECT
  ams.ProductID,
  ams.ProductName,
  cs.current_quantity,
  ams.avg_monthly_sales_quantity,
  CASE
    WHEN cs.current_quantity < ams.avg_monthly_sales_quantity THEN 'Stock-Out Risk'
    ELSE 'No Risk'
  END AS stock_out_prediction
FROM avg_monthly_sales AS ams
JOIN current_stock AS cs ON ams.ProductID = cs.ProductID;

SELECT * FROM invt.vw_stock_out_prediction;

In [0]:
%sql
SELECT 
ph.productName,
sum(ph.Price * ph.Quantity) as purchase_amount,
sum(os.Quantity * ph.Price) as revenue
FROM invt.purchasehistory as ph
Join invt.order_status as os on ph.ProductName = os.ProductName
group by ph.productName
order by revenue desc



In [0]:
%sql
-- •	Pending order tracking with fulfillment time analysis 
CREATE OR REPLACE VIEW invt.vw_pending_orders AS
with pending as (
select 
os.ProductName,
os.OrderStatus,
os.Quantity,
cast (os.statusdate as date) as statusdate
from invt.order_status as os
where os.OrderStatus = 'Pending'
)
SELECT
  p.ProductName,
  p.OrderStatus,
  p.Quantity,
  p.statusdate,
CASE
    WHEN DATEDIFF(CURRENT_DATE(), p.statusdate) <= 10 THEN '0 - 10 days'
    WHEN DATEDIFF(CURRENT_DATE(), p.statusdate) <= 30 THEN '11 - 30 days'
    else '30+ days'
  end as pending_days
FROM pending as p;

select * from invt.vw_pending_orders

In [0]:
%sql
-- •	Product performance ranking within categories using RANK/DENSE_RANK from invt
CREATE OR REPLACE VIEW invt.vw_product_performance AS
SELECT
  dp.ProductCategory,
  dp.ProductName,
  SUM(ft.ProductQuantity * ft.Price) AS revenue,
  RANK() OVER (PARTITION BY dp.ProductCategory ORDER BY SUM(ft.ProductQuantity * ft.Price) DESC) AS revenue_rank,
  DENSE_RANK() OVER (PARTITION BY dp.ProductCategory ORDER BY SUM(ft.ProductQuantity * ft.Price) DESC) AS revenue_dense_rank
FROM invt.factinventoryproduct AS ft
JOIN invt.dim_product AS dp ON ft.ProductID = dp.ProductID
GROUP BY dp.ProductCategory, dp.ProductName
ORDER BY dp.ProductCategory, revenue_rank;

select * from invt.vw_product_performance

In [0]:
%sql
-- • Customer lifetime value calculations from invt
create or replace view invt.vw_customer_lifetime_value as
with customer_life as (
  select 
  ft.CustomerID,
  MIN( cast (dd.Date as date)) as first_purchase,
  MAX(cast (dd.Date as date) ) as last_purchase
  from invt.factinventoryproduct as ft
  join invt.dim_date as dd on ft.orderDateID = dd.DateID
  group by ft.CustomerID
)
SELECT 
customerid,
first_purchase,
last_purchase,
Datediff(last_purchase , first_purchase) lifespan_day
FROM customer_life 
group by customerid, first_purchase, last_purchase;

select * from invt.vw_customer_lifetime_value


In [0]:
%sql
-- # •	Warehouse efficiency comparison by inventory turnover
create or replace view invt.vw_warehouse_efficiency as
with warehouse_comparison as (
SELECT 
dw.warehousename as warehouse_name ,
sum(ph.Quantity * ph.Price) as inventory_value,
sum(os.Quantity) / avg(ph.Quantity) as turnover_value
from invt.factinventoryproduct as ft
join invt.purchasehistory as ph on ft.Price = ph.Price
join invt.order_status as os on ph.ProductName = os.ProductName
join invt.dim_warehouse as dw on ft.WareHouseID = dw.WareHouseID
group by dw.warehousename
)
SELECT
wc.warehouse_name ,
wc.inventory_value,
wc.turnover_value 
FROM warehouse_comparison as wc;

SELECT * from invt.Vw_warehouse_efficiency



In [0]:
%sql
-- # • Customer acquisition cohort analysis by year from invt
create or replace view invt.vw_customer_acquisition as
SELECT
  dd.Year AS acquisition_year,
  COUNT(DISTINCT ft.CustomerID) AS customers_acquired
FROM invt.factinventoryproduct AS ft
JOIN invt.dim_date AS dd ON ft.orderDateID = dd.DateID
GROUP BY dd.Year
ORDER BY dd.Year;

select * from invt.vw_customer_acquisition

In [0]:
%sql
-- •	Pending order tracking with location details
create or replace view invt.vw_pending_orders_location AS
SELECT
  os.ProductName,
  os.OrderStatus,
  os.Quantity,
  cast (os.statusdate as date ) as statusdate,
  dc.CustomerCity AS location
FROM invt.order_status AS os
JOIN invt.dim_customer AS dc ON os.CustomerName = dc.CustomerName
WHERE os.OrderStatus = 'Pending';

select * from invt.vw_pending_orders_location

In [0]:
%sql
-- Order status workflow (Pending, Delivered, Shipped)
create or replace view invt.vw_order_status_workflow AS
SELECT
  
  os.ProductName,
  os.OrderStatus,
  os.Quantity,
  cast (os.statusdate as date) as statusdate
FROM invt.order_status AS os
WHERE os.OrderStatus IN ('Pending', 'Delivered', 'Shipped');

select * from invt.vw_order_status_workflow

In [0]:
%sql
-- 	Fulfillment time analysis by geographic region
CREATE OR REPLACE VIEW invt.vw_fulfillment_time_region AS
SELECT 
 cast (bmd.OrderDate as date) as orderDate ,
cast( bmd.DeliveryDate as date ) as DeliveryDate,
cast( bmd.DeliveryDate as date ) -  cast( bmd.OrderDate  as date) as Fulfillment_time
FROM invt.factinventoryproduct as ft
join invt.dim_date as dd on ft.orderDateID = dd.DateID
join invt.bharat_mart_dataset as bmd on ft.ProductQuantity = ft.ProductQuantity;

select * from invt.vw_fulfillment_time_region